Stage 1 : Data Ingestion

In [15]:
import pandas as pd

# Load datasets
deliveries_df = pd.read_csv("../datasets/deliveries.csv")
matches_df = pd.read_csv("../datasets/matches.csv")

# Inspect shape
print("deliveries_df shape:", deliveries_df.shape)
print("matches_df shape:", matches_df.shape)

# Inspect columns
print("\nDeliveries columns:")
print(deliveries_df.columns.tolist())

print("\nMatches columns:")
print(matches_df.columns.tolist())

# Inspect data types
print("\nDeliveries dtypes:")
print(deliveries_df.dtypes)

print("\nMatches dtypes:")
print(matches_df.dtypes)

# Quick preview
display(deliveries_df.head())
display(matches_df.head())

deliveries_df shape: (260920, 17)
matches_df shape: (1095, 20)

Deliveries columns:
['match_id', 'inning', 'batting_team', 'bowling_team', 'over', 'ball', 'batter', 'bowler', 'non_striker', 'batsman_runs', 'extra_runs', 'total_runs', 'extras_type', 'is_wicket', 'player_dismissed', 'dismissal_kind', 'fielder']

Matches columns:
['id', 'season', 'city', 'date', 'match_type', 'player_of_match', 'venue', 'team1', 'team2', 'toss_winner', 'toss_decision', 'winner', 'result', 'result_margin', 'target_runs', 'target_overs', 'super_over', 'method', 'umpire1', 'umpire2']

Deliveries dtypes:
match_id            int64
inning              int64
batting_team          str
bowling_team          str
over                int64
ball                int64
batter                str
bowler                str
non_striker           str
batsman_runs        int64
extra_runs          int64
total_runs          int64
extras_type           str
is_wicket           int64
player_dismissed      str
dismissal_kind        

,match_id,inning,batting_team,bowling_team,over,ball,batter,bowler,non_striker,batsman_runs,extra_runs,total_runs,extras_type,is_wicket,player_dismissed,dismissal_kind,fielder
0,335982,1,Kolkata Knight Riders,Royal Challengers Bangalore,0,1,SC Ganguly,P Kumar,BB McCullum,0,1,1,legbyes,0,NaN,NaN,NaN
1,335982,1,Kolkata Knight Riders,Royal Challengers Bangalore,0,2,BB McCullum,P Kumar,SC Ganguly,0,0,0,NaN,0,NaN,NaN,NaN
2,335982,1,Kolkata Knight Riders,Royal Challengers Bangalore,0,3,BB McCullum,P Kumar,SC Ganguly,0,1,1,wides,0,NaN,NaN,NaN
3,335982,1,Kolkata Knight Riders,Royal Challengers Bangalore,0,4,BB McCullum,P Kumar,SC Ganguly,0,0,0,NaN,0,NaN,NaN,NaN
4,335982,1,Kolkata Knight Riders,Royal Challengers Bangalore,0,5,BB McCullum,P Kumar,SC Ganguly,0,0,0,NaN,0,NaN,NaN,NaN


,id,season,city,date,match_type,player_of_match,venue,team1,team2,toss_winner,toss_decision,winner,result,result_margin,target_runs,target_overs,super_over,method,umpire1,umpire2
0,335982,2007/08,Bangalore,2008-04-18,League,BB McCullum,M Chinnaswamy Stadium,Royal Challengers Bangalore,Kolkata Knight Riders,Royal Challengers Bangalore,field,Kolkata Knight Riders,runs,140.0,223.0,20.0,N,NaN,Asad Rauf,RE Koertzen
1,335983,2007/08,Chandigarh,2008-04-19,League,MEK Hussey,"Punjab Cricket Association Stadium, Mohali",Kings XI Punjab,Chennai Super Kings,Chennai Super Kings,bat,Chennai Super Kings,runs,33.0,241.0,20.0,N,NaN,MR Benson,SL Shastri
2,335984,2007/08,Delhi,2008-04-19,League,MF Maharoof,Feroz Shah Kotla,Delhi Daredevils,Rajasthan Royals,Rajasthan Royals,bat,Delhi Daredevils,wickets,9.0,130.0,20.0,N,NaN,Aleem Dar,GA Pratapkumar
3,335985,2007/08,Mumbai,2008-04-20,League,MV Boucher,Wankhede Stadium,Mumbai Indians,Royal Challengers Bangalore,Mumbai Indians,bat,Royal Challengers Bangalore,wickets,5.0,166.0,20.0,N,NaN,SJ Davis,DJ Harper
4,335986,2007/08,Kolkata,2008-04-20,League,DJ Hussey,Eden Gardens,Kolkata Knight Riders,Deccan Chargers,Deccan Chargers,bat,Kolkata Knight Riders,wickets,5.0,111.0,20.0,N,NaN,BF Bowden,K Hariharan


Stage 2 : Data Cleaning & Validation

In [32]:
deliveries_clean = deliveries_df.copy()
matches_clean = matches_df.copy()

# Check missing values
print("Missing values in deliveries:")
print(deliveries_clean.isna().sum()[deliveries_clean.isna().sum() > 0])

print("\nMissing values in matches:")
print(matches_clean.isna().sum()[matches_clean.isna().sum() > 0])

# Clean text columns
delivery_cat_cols = deliveries_clean.select_dtypes(include=["object", "string"]).columns
match_cat_cols = matches_clean.select_dtypes(include=["object", "string"]).columns

for col in delivery_cat_cols:
    deliveries_clean[col] = deliveries_clean[col].astype("string").str.strip()

for col in match_cat_cols:
    matches_clean[col] = matches_clean[col].astype("string").str.strip()

# Convert date
matches_clean["date"] = pd.to_datetime(matches_clean["date"], errors="coerce")

# Correct numeric columns
delivery_numeric_cols = ["match_id", "inning", "over", "ball", "batsman_runs", "extra_runs", "total_runs", "is_wicket"]
match_numeric_cols = ["id", "result_margin", "target_runs", "target_overs"]

for col in delivery_numeric_cols:
    deliveries_clean[col] = pd.to_numeric(deliveries_clean[col], errors="coerce")

for col in match_numeric_cols:
    matches_clean[col] = pd.to_numeric(matches_clean[col], errors="coerce")

# Validate match ID alignment
delivery_ids = set(deliveries_clean["match_id"].dropna().astype(int).unique())
match_ids = set(matches_clean["id"].dropna().astype(int).unique())

print("\nIDs only in deliveries:", len(delivery_ids - match_ids))
print("IDs only in matches:", len(match_ids - delivery_ids))

# Keep only aligned rows
deliveries_clean = deliveries_clean[deliveries_clean["match_id"].isin(match_ids)].copy()
matches_clean = matches_clean[matches_clean["id"].isin(delivery_ids)].copy()

# Final check
print("\nCleaned shapes:")
print("deliveries_clean:", deliveries_clean.shape)
print("matches_clean:", matches_clean.shape)

Missing values in deliveries:
extras_type         246795
player_dismissed    247970
dismissal_kind      247970
fielder             251566
dtype: int64

Missing values in matches:
city                 51
player_of_match       5
winner                5
result_margin        19
target_runs           3
target_overs          3
method             1074
dtype: int64

IDs only in deliveries: 0
IDs only in matches: 0

Cleaned shapes:
deliveries_clean: (260920, 17)
matches_clean: (1095, 20)


Stage 3 : Data Transformation

In [34]:
# Stage 3: Data Transformation
ipl_df = deliveries_clean.merge(matches_clean, left_on="match_id", right_on="id", how="inner")
ipl_df.columns = ipl_df.columns.str.strip().str.lower()


# Total runs per ball
ipl_df["runs_per_ball"] = ipl_df["batsman_runs"] + ipl_df["extra_runs"]
print(ipl_df.columns.to_list())

print("Unified DataFrame shape:", ipl_df.shape)
display(ipl_df.head())

['match_id', 'inning', 'batting_team', 'bowling_team', 'over', 'ball', 'batter', 'bowler', 'non_striker', 'batsman_runs', 'extra_runs', 'total_runs', 'extras_type', 'is_wicket', 'player_dismissed', 'dismissal_kind', 'fielder', 'id', 'season', 'city', 'date', 'match_type', 'player_of_match', 'venue', 'team1', 'team2', 'toss_winner', 'toss_decision', 'winner', 'result', 'result_margin', 'target_runs', 'target_overs', 'super_over', 'method', 'umpire1', 'umpire2', 'runs_per_ball']
Unified DataFrame shape: (260920, 38)


,match_id,inning,batting_team,bowling_team,over,ball,batter,bowler,non_striker,batsman_runs,...,winner,result,result_margin,target_runs,target_overs,super_over,method,umpire1,umpire2,runs_per_ball
0,335982,1,Kolkata Knight Riders,Royal Challengers Bangalore,0,1,SC Ganguly,P Kumar,BB McCullum,0,...,Kolkata Knight Riders,runs,140.0,223.0,20.0,N,<NA>,Asad Rauf,RE Koertzen,1
1,335982,1,Kolkata Knight Riders,Royal Challengers Bangalore,0,2,BB McCullum,P Kumar,SC Ganguly,0,...,Kolkata Knight Riders,runs,140.0,223.0,20.0,N,<NA>,Asad Rauf,RE Koertzen,0
2,335982,1,Kolkata Knight Riders,Royal Challengers Bangalore,0,3,BB McCullum,P Kumar,SC Ganguly,0,...,Kolkata Knight Riders,runs,140.0,223.0,20.0,N,<NA>,Asad Rauf,RE Koertzen,1
3,335982,1,Kolkata Knight Riders,Royal Challengers Bangalore,0,4,BB McCullum,P Kumar,SC Ganguly,0,...,Kolkata Knight Riders,runs,140.0,223.0,20.0,N,<NA>,Asad Rauf,RE Koertzen,0
4,335982,1,Kolkata Knight Riders,Royal Challengers Bangalore,0,5,BB McCullum,P Kumar,SC Ganguly,0,...,Kolkata Knight Riders,runs,140.0,223.0,20.0,N,<NA>,Asad Rauf,RE Koertzen,0


Stage 4 : Core Analysis

In [ ]:
# Stage 4: Core Analysis

analysis_df = ipl_df.copy()
analysis_df["is_legal_delivery"] = ~analysis_df["extras_type"].isin(["wides", "noballs"])

# 1) Total runs per match
runs_per_match = analysis_df.groupby(["match_id", "season", "date"], as_index=False)["total_runs"].sum()
runs_per_match = runs_per_match.rename(columns={"total_runs": "runs_scored"}).sort_values("runs_scored", ascending=False)

# 2) Runs per team per match
team_scores = analysis_df.groupby(["match_id", "batting_team"], as_index=False)["total_runs"].sum()
team_scores = team_scores.rename(columns={"total_runs": "team_runs"}).sort_values(["match_id", "team_runs"], ascending=[True, False])

# 3) Top 10 batters
batting_runs = analysis_df.groupby("batter", as_index=False)["batsman_runs"].sum()
top_batters = batting_runs.rename(columns={"batsman_runs": "total_runs"}).sort_values("total_runs", ascending=False).head(10)

# 4) Strike rate of batters
batting_summary = analysis_df[analysis_df["is_legal_delivery"]].groupby("batter", as_index=False).agg(
    runs_scored=("batsman_runs", "sum"),
    balls_faced=("batsman_runs", "size")
)
batting_summary["strike_rate"] = (batting_summary["runs_scored"] / batting_summary["balls_faced"]) * 100
strike_rate = batting_summary.sort_values("strike_rate", ascending=False)

# 5) Top 10 bowlers by economy
bowling_runs = analysis_df.groupby("bowler", as_index=False)["total_runs"].sum().rename(columns={"total_runs": "runs_conceded"})
bowling_balls = analysis_df[analysis_df["is_legal_delivery"]].groupby("bowler", as_index=False).size().rename(columns={"size": "balls_bowled"})
economy = bowling_runs.merge(bowling_balls, on="bowler", how="left")
economy["overs_bowled"] = economy["balls_bowled"] / 6
economy["economy_rate"] = economy["runs_conceded"] / economy["overs_bowled"]
economy = economy.sort_values("economy_rate", ascending=True).head(10)

# 6) Most consistent batters
batter_match_runs = analysis_df.groupby(["batter", "match_id"], as_index=False)["batsman_runs"].sum()
consistent_batters = batter_match_runs.groupby("batter", as_index=False).agg(
    avg_runs_per_match=("batsman_runs", "mean"),
    matches_played=("match_id", "nunique")
)
consistent_batters = consistent_batters[consistent_batters["matches_played"] >= 10]
consistent_batters = consistent_batters.sort_values(["avg_runs_per_match", "matches_played"], ascending=[False, False])

# 7) Highest individual score in a match
highest_individual_score = batter_match_runs.sort_values("batsman_runs", ascending=False).head(1)

# 8) Boundary analysis
boundary_analysis = analysis_df.groupby("batter", as_index=False).agg(
    total_fours=("batsman_runs", lambda x: (x == 4).sum()),
    total_sixes=("batsman_runs", lambda x: (x == 6).sum())
)
boundary_analysis["total_boundaries"] = boundary_analysis["total_fours"] + boundary_analysis["total_sixes"]
boundary_analysis["boundary_runs"] = (boundary_analysis["total_fours"] * 4) + (boundary_analysis["total_sixes"] * 6)
total_fours = boundary_analysis["total_fours"].sum()
total_sixes = boundary_analysis["total_sixes"].sum()
top_boundary_players = boundary_analysis.sort_values("total_boundaries", ascending=False).head(10)

# 9) Boundary percentage
batting_runs_total = batting_runs.rename(columns={"batsman_runs": "total_runs"})
boundary_percentage = boundary_analysis.merge(batting_runs_total, on="batter", how="left")
boundary_percentage["boundary_percentage"] = (boundary_percentage["boundary_runs"] / boundary_percentage["total_runs"]) * 100
boundary_percentage = boundary_percentage.sort_values("boundary_percentage", ascending=False)

# 10) Dot ball analysis
dot_balls = analysis_df[analysis_df["total_runs"] == 0]
total_dot_balls = len(dot_balls)
dot_ball_bowlers = dot_balls.groupby("bowler", as_index=False).size().rename(columns={"size": "dot_balls"}).sort_values("dot_balls", ascending=False)

# 11) Runs per over analysis
runs_per_over = analysis_df.groupby("over", as_index=False)["total_runs"].mean().rename(columns={"total_runs": "average_runs"}).sort_values("over")
high_scoring_overs = runs_per_over.sort_values("average_runs", ascending=False).head(5)

# 12) Powerplay performance (overs 1-6)
powerplay_df = analysis_df[analysis_df["over"].between(1, 6)]
powerplay_total_runs = powerplay_df["total_runs"].sum()
best_powerplay_teams = powerplay_df.groupby("batting_team", as_index=False)["total_runs"].sum().rename(columns={"total_runs": "powerplay_runs"}).sort_values("powerplay_runs", ascending=False)

# 13) Death overs performance (overs 16-20)
death_overs_df = analysis_df[analysis_df["over"].between(16, 20)]
death_overs = death_overs_df.groupby("batting_team", as_index=False)["total_runs"].sum().rename(columns={"total_runs": "death_over_runs"}).sort_values("death_over_runs", ascending=False)
best_death_batters = death_overs_df.groupby("batter", as_index=False)["total_runs"].sum().rename(columns={"total_runs": "death_over_runs"}).sort_values("death_over_runs", ascending=False)

# 14) Run distribution per inning
inning_runs = analysis_df.groupby("inning", as_index=False).agg(
    total_runs=("total_runs", "sum"),
    average_runs=("total_runs", "mean")
)

# 15) Toss impact analysis
match_team_runs = analysis_df.groupby(["match_id", "batting_team"], as_index=False)["total_runs"].sum().rename(columns={"total_runs": "team_runs"})
toss_impact = match_team_runs.merge(matches_clean[["id", "toss_winner", "winner"]], left_on="match_id", right_on="id", how="left")
toss_impact["is_toss_winner_team"] = toss_impact["batting_team"] == toss_impact["toss_winner"]
toss_winner_avg_runs = toss_impact[toss_impact["is_toss_winner_team"]]["team_runs"].mean()
opponent_avg_runs = toss_impact[~toss_impact["is_toss_winner_team"]]["team_runs"].mean()

# 16) Player of match contribution
top_batter_per_match = batter_match_runs.sort_values(["match_id", "batsman_runs"], ascending=[True, False]).drop_duplicates("match_id")
top_batter_per_match = top_batter_per_match.rename(columns={"batter": "top_batter", "batsman_runs": "top_batter_runs"})
player_of_match_contribution = matches_clean[["id", "player_of_match", "winner"]].merge(top_batter_per_match, left_on="id", right_on="match_id", how="left")
player_of_match_contribution["player_of_match_was_top_scorer"] = player_of_match_contribution["player_of_match"] == player_of_match_contribution["top_batter"]

# 17) Venue-wise analysis
match_runs = analysis_df.groupby("match_id", as_index=False)["total_runs"].sum().rename(columns={"total_runs": "match_runs"})
venue_analysis = matches_clean.merge(match_runs, left_on="id", right_on="match_id", how="left")
venue_summary = venue_analysis.groupby("venue", as_index=False).agg(
    total_matches=("id", "nunique"),
    average_runs=("match_runs", "mean")
).sort_values("average_runs", ascending=False)

# 18) City-wise scoring trends
city_summary = venue_analysis.groupby("city", as_index=False).agg(
    total_matches=("id", "nunique"),
    average_runs=("match_runs", "mean")
).sort_values("average_runs", ascending=False)

# 19) Season-wise run trends
season_summary = venue_analysis.groupby("season", as_index=False).agg(total_runs=("match_runs", "sum")).sort_values("season")
season_summary["growth_rate"] = season_summary["total_runs"].pct_change() * 100

# 20) Winning team analysis
predicted_winner = match_team_runs.sort_values(["match_id", "team_runs"], ascending=[True, False]).drop_duplicates("match_id")
predicted_winner = predicted_winner.rename(columns={"batting_team": "predicted_winner"})
winning_team_analysis = matches_clean[["id", "winner"]].merge(predicted_winner[["match_id", "predicted_winner"]], left_on="id", right_on="match_id", how="left")
winning_team_analysis["matched_actual_winner"] = winning_team_analysis["winner"] == winning_team_analysis["predicted_winner"]
winning_team_accuracy = winning_team_analysis["matched_actual_winner"].mean() * 100

# Quick view of key results
display(runs_per_match.head())
display(top_batters)
display(strike_rate.head(10))
display(economy)
display(consistent_batters.head(10))

,match_id,season,date,runs_scored
1053,1426268,2024,2024-04-15,549
1031,1422126,2024,2024-03-27,523
1065,1426280,2024,2024-04-26,523
1066,1426281,2024,2024-04-27,504
146,419137,2009/10,2010-04-03,469


,batter,total_runs
631,V Kohli,8014
512,S Dhawan,6769
477,RG Sharma,6630
147,DA Warner,6567
546,SK Raina,5536
374,MS Dhoni,5243
30,AB de Villiers,5181
124,CH Gayle,4997
501,RV Uthappa,4954
282,KD Karthik,4843


,batter,runs_scored,balls_faced,strike_rate
312,L Wood,9,3,300.000000
97,B Stanlake,5,2,250.000000
234,J Fraser-McGurk,324,139,233.093525
461,R Sai Kishore,13,6,216.666667
629,Umar Gul,39,19,205.263158
497,RS Sodhi,4,2,200.000000
465,R Shepherd,115,62,185.483871
410,Naman Dhir,140,78,179.487179
433,PD Salt,652,368,177.173913
583,Shahid Afridi,81,46,176.086957


,bowler,runs_conceded,balls_bowled,overs_bowled,economy_rate
24,AC Gilchrist,0,1,0.166667,0.000000
364,R Ravindra,7,12,2.000000,3.500000
317,NB Singh,18,24,4.000000,4.500000
460,Sachin Baby,8,10,1.666667,4.800000
38,AM Rahane,5,6,1.000000,5.000000
125,DJ Thornely,40,42,7.000000,5.714286
260,M Manhas,42,42,7.000000,6.000000
453,SS Mundhe,6,6,1.000000,6.000000
245,LA Carseldine,6,6,1.000000,6.000000
296,MW Short,25,24,4.000000,6.250000


,batter,avg_runs_per_match,matches_played
170,DP Conway,42.000000,22
96,B Sai Sudharsan,41.360000,25
289,KL Rahul,38.434426,122
319,LMP Simmons,37.206897,29
473,RD Gaikwad,36.615385,65
542,SE Marsh,36.072464,69
214,HM Amla,36.062500,16
147,DA Warner,35.690217,184
124,CH Gayle,35.439716,141
365,ML Hayden,34.593750,32


Stage 5 : Derived Insights

In [ ]:
# Stage 5: Derived Insights
most_consistent_batter = consistent_batters.iloc[0]["batter"] if not consistent_batters.empty else "N/A"
best_death_team = death_overs.iloc[0]["batting_team"] if not death_overs.empty else "N/A"
high_scoring_venue = venue_summary.iloc[0]["venue"] if not venue_summary.empty else "N/A"

print("Most consistent batter:", most_consistent_batter)
print("Best death-over team:", best_death_team)
print("High-scoring venue:", high_scoring_venue)

print("\nShort observations:")
print("- Consistent batters are the ones with strong average runs across many matches.")
print("- Teams that score well in death overs usually finish innings strongly.")
print("- High-scoring venues tend to produce larger match totals.")

Most consistent batter: DP Conway
Best death-over team: Mumbai Indians
High-scoring venue: Dr. Y.S. Rajasekhara Reddy ACA-VDCA Cricket Stadium, Visakhapatnam

Short observations:
- Consistent batters are the ones with strong average runs across many matches.
- Teams that score well in death overs usually finish innings strongly.
- High-scoring venues tend to produce larger match totals.


Stage 6 : Reporting

In [ ]:
# Stage 6: Reporting
runs_per_match_report = runs_per_match.rename(columns={"runs_scored": "total_runs"}).sort_values("total_runs", ascending=False)
top_batters_report = top_batters.rename(columns={"total_runs": "runs_scored"}).sort_values("runs_scored", ascending=False)
strike_rate_report = strike_rate.rename(columns={"runs_scored": "runs_scored", "balls_faced": "balls_faced"})
economy_report = economy.rename(columns={"runs_conceded": "runs_conceded", "balls_bowled": "balls_bowled", "overs_bowled": "overs_bowled", "economy_rate": "economy_rate"})
team_scores_report = team_scores.rename(columns={"team_runs": "total_runs"})
death_overs_report = death_overs.rename(columns={"death_over_runs": "runs_scored"})

print("Reporting tables ready:")
display(runs_per_match_report.head())
display(top_batters_report)
display(strike_rate_report.head(10))
display(economy_report)
display(team_scores_report.head())
display(death_overs_report.head(10))

Reporting tables ready:


,match_id,season,date,total_runs
1053,1426268,2024,2024-04-15,549
1031,1422126,2024,2024-03-27,523
1065,1426280,2024,2024-04-26,523
1066,1426281,2024,2024-04-27,504
146,419137,2009/10,2010-04-03,469


,batter,runs_scored
631,V Kohli,8014
512,S Dhawan,6769
477,RG Sharma,6630
147,DA Warner,6567
546,SK Raina,5536
374,MS Dhoni,5243
30,AB de Villiers,5181
124,CH Gayle,4997
501,RV Uthappa,4954
282,KD Karthik,4843


,batter,runs_scored,balls_faced,strike_rate
312,L Wood,9,3,300.000000
97,B Stanlake,5,2,250.000000
234,J Fraser-McGurk,324,139,233.093525
461,R Sai Kishore,13,6,216.666667
629,Umar Gul,39,19,205.263158
497,RS Sodhi,4,2,200.000000
465,R Shepherd,115,62,185.483871
410,Naman Dhir,140,78,179.487179
433,PD Salt,652,368,177.173913
583,Shahid Afridi,81,46,176.086957


,bowler,runs_conceded,balls_bowled,overs_bowled,economy_rate
24,AC Gilchrist,0,1,0.166667,0.000000
364,R Ravindra,7,12,2.000000,3.500000
317,NB Singh,18,24,4.000000,4.500000
460,Sachin Baby,8,10,1.666667,4.800000
38,AM Rahane,5,6,1.000000,5.000000
125,DJ Thornely,40,42,7.000000,5.714286
260,M Manhas,42,42,7.000000,6.000000
453,SS Mundhe,6,6,1.000000,6.000000
245,LA Carseldine,6,6,1.000000,6.000000
296,MW Short,25,24,4.000000,6.250000


,match_id,batting_team,total_runs
0,335982,Kolkata Knight Riders,222
1,335982,Royal Challengers Bangalore,82
2,335983,Chennai Super Kings,240
3,335983,Kings XI Punjab,207
4,335984,Delhi Daredevils,132


,batting_team,runs_scored
10,Mumbai Indians,9598
0,Chennai Super Kings,9061
16,Royal Challengers Bangalore,8417
8,Kolkata Knight Riders,8053
13,Rajasthan Royals,7281
18,Sunrisers Hyderabad,6237
6,Kings XI Punjab,6227
3,Delhi Daredevils,5043
2,Delhi Capitals,3141
1,Deccan Chargers,2539


Stage 7 : Data Export

In [2]:
# Stage 7: Data Export

import os

output_dir = "../output"
os.makedirs(output_dir, exist_ok=True)

runs_per_match_report.to_csv(os.path.join(output_dir, "runs_per_match.csv"), index=False)
top_batters_report.to_csv(os.path.join(output_dir, "top_batters.csv"), index=False)
strike_rate_report.to_csv(os.path.join(output_dir, "strike_rate.csv"), index=False)
economy_report.to_csv(os.path.join(output_dir, "economy.csv"), index=False)
team_scores_report.to_csv(os.path.join(output_dir, "team_scores.csv"), index=False)
death_overs_report.to_csv(os.path.join(output_dir, "death_overs.csv"), index=False)

try:
    with pd.ExcelWriter(os.path.join(output_dir, "ipl_analysis.xlsx")) as writer:
        runs_per_match_report.to_excel(writer, sheet_name="Runs per Match", index=False)
        top_batters_report.to_excel(writer, sheet_name="Top Batters", index=False)
        strike_rate_report.to_excel(writer, sheet_name="Strike Rate", index=False)
        economy_report.to_excel(writer, sheet_name="Economy", index=False)
        team_scores_report.to_excel(writer, sheet_name="Team Scores", index=False)
        death_overs_report.to_excel(writer, sheet_name="Death Overs", index=False)
    print("Excel file saved.")
except ModuleNotFoundError:
    print("CSV files saved. Excel export skipped because openpyxl is not installed.")

print("Files saved in the output folder.")

NameError: name 'runs_per_match_report' is not defined